# VARIMAX where X is vector of `r.grf.y_RANK` and Y is vector of `r.gt.z_RANK`

In [1]:
import copy
import functools
import glob
import pickle
import scipy
import sklearn
import warnings

warnings.filterwarnings('ignore')

import itertools
import graphviz as gr
import numpy as np
import os
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm


from matplotlib import style
from matplotlib import pyplot as plt
from pprint import pprint
from scipy import stats, special
from sklearn import datasets, mixture
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

import plotly.express as px
import plotly.figure_factory as ff

%matplotlib inline

style.use("fivethirtyeight")
pd.set_option("display.max_columns", 6)

np.random.seed(0)
pd.set_option('display.max_columns', None)

In [2]:
#!pip install --upgrade pandas --user

### File Locations

In [3]:
DATA_FOLDER = "data"
FIGURES_FOLDER = "figures"
OUTPUT_FOLDER = "output"

DATA_FOLDER = os.path.join("..",DATA_FOLDER)
FIGURES_FOLDER = os.path.join("..",FIGURES_FOLDER)
OUTPUT_FOLDER = os.path.join("..",OUTPUT_FOLDER)


ALL_BACKTEST_PATH = os.path.join(DATA_FOLDER,"all_backtest.csv")

CDPHE_FOLDER = os.path.join(OUTPUT_FOLDER,"CDPHE")
if not os.path.exists(CDPHE_FOLDER):
    os.makedirs(CDPHE_FOLDER)

### Outbreak Matrix

In [4]:
OUTBREAK_MATRIX_PATH = os.path.join(DATA_FOLDER,"colorado_outbreak_matrix.csv")
outbreak_matrix = pd.read_csv(OUTBREAK_MATRIX_PATH,parse_dates=True, index_col=0)
# Generate changepoints
changepoint_matrix = outbreak_matrix.diff() 
#nunique = changepoint_matrix.nunique()
#cols_to_drop = nunique[nunique == 1].index
#changepoint_matrix = changepoint_matrix.drop(cols_to_drop, axis=1)
changepoint_matrix = changepoint_matrix.stack().reset_index(level=[0,1])
changepoint_matrix = changepoint_matrix.rename(columns={"level_0":"date.y","level_1":"fips",0:"changepoint"})
changepoint_matrix = changepoint_matrix[changepoint_matrix["changepoint"]==1]

CDPHE_investigated = changepoint_matrix[changepoint_matrix["changepoint"]==1]
CDPHE_investigated["fips"] = CDPHE_investigated["fips"].astype(int).to_numpy().flatten()
CDPHE_investigated["date_outbreak_declared"] = CDPHE_investigated["date.y"].to_numpy().flatten()
CDPHE_investigated = CDPHE_investigated.reset_index()
CDPHE_investigated = CDPHE_investigated[["fips","date.y"]]
# Decision Point should be when it is white
CDPHE_investigated["chosen_by_CDPHE"] = 1
CDPHE_investigated["date.y"] = CDPHE_investigated["date.y"]+pd.DateOffset(days=-1)
CDPHE_investigated.head()

,fips,date.y,chosen_by_CDPHE
0,8005,2020-03-17,1
1,8013,2020-03-18,1
2,8015,2020-03-19,1
3,8001,2020-03-22,1
4,8059,2020-03-22,1


### We now generate the full spectrum of decision spaces

In [5]:
investigation_df = outbreak_matrix.stack().reset_index(level=[0,1])
investigation_df = investigation_df.rename(columns={"level_0":"date.y","level_1":"fips",0:"under_investigation"})
investigation_df["fips"] = investigation_df["fips"].astype(int)

### What was the max, min, median, mean of number of counties under investigation in general?

In [6]:
investigation_df.groupby("date.y")["under_investigation"].agg(["sum"]).agg(["max","min","mean","median"])

,sum
max,46.000000
min,1.000000
mean,28.597378
median,27.000000


In [7]:
investigation_df["fips"]

0        8001
1        8003
2        8005
3        8007
4        8009
         ... 
34171    8117
34172    8119
34173    8121
34174    8123
34175    8125
Name: fips, Length: 34176, dtype: int32

### Obtain the backtest_data so that we can merge w/ investigation status to obtain decision space

In [8]:
all_backtest_df = pd.read_csv(ALL_BACKTEST_PATH, parse_dates=True)
# For this analysis:
# x = 7 days ago
# y = now
# z = 7 days later
all_backtest_df = all_backtest_df.dropna(subset=["log_rolled_cases.y"])
# Generate dates
all_backtest_df["date.y"] = pd.to_datetime(all_backtest_df["date.y"])
all_backtest_df["date.x"] = all_backtest_df["date.y"] +pd.DateOffset(days=-6)
all_backtest_df["date.z"] = all_backtest_df["date.y"] +pd.DateOffset(days=6)
# Generate days from start 
all_backtest_df["days_from_start.x"] = all_backtest_df["days_from_start.y"] - 7
all_backtest_df["days_from_start.z"] = all_backtest_df["days_from_start.y"] + 7
# Sort by date within each county
all_backtest_df = all_backtest_df.groupby("fips").apply(lambda x: x.sort_values(by="date.y"))
all_backtest_df = all_backtest_df.reset_index(drop=True)
# Generate Case data for each county
all_backtest_df["log_rolled_cases.x"] = all_backtest_df.groupby("fips")["log_rolled_cases.y"].shift(7)
all_backtest_df["log_rolled_cases.z"] = all_backtest_df.groupby("fips")["log_rolled_cases.y"].shift(-7)

# Obtain ground truth growth rates one week later
all_backtest_df["r.gt.z"] = all_backtest_df["r.gt.y"].shift(-7)
# Obtain current and one week later rate prediction
all_backtest_df["r.grf.x"] = all_backtest_df["r.grf"]
all_backtest_df["r.grf.y"] = all_backtest_df.groupby("fips")["r.grf"].shift(-7)
all_backtest_df["r.grf.z"] = all_backtest_df.groupby("fips")["r.grf"].shift(-14)

# Drop NA
all_backtest_df = all_backtest_df.dropna(subset=["r.gt.y"])
all_backtest_df = all_backtest_df.dropna(subset=["r.gt.z"])
all_backtest_df = all_backtest_df.dropna(subset=["r.grf.y"])

# Obtain states so that we can rank counties within state for each county
county_fips_master_df = pd.read_csv(os.path.join(OUTPUT_FOLDER,"county_fips_master.csv"))
all_backtest_df = pd.merge(left=all_backtest_df, right = county_fips_master_df[["fips","state_abbr"]],how="left",left_on="fips",right_on="fips")


In [9]:
decision_space = pd.merge(left=all_backtest_df, right=investigation_df, how="inner", on=["fips","date.y"])
decision_space = decision_space.sort_values(by=["date.y","fips"])
decision_space.head()

,fips,county,r.lm,predicted.lm,date.y,days_from_start.y,log_rolled_cases.y,r.grf,tau.variance,tau.upr,tau.lwr,predicted.grf,date_delta,r.gt.y,date.x,date.z,days_from_start.x,days_from_start.z,log_rolled_cases.x,log_rolled_cases.z,r.gt.z,r.grf.x,r.grf.y,r.grf.z,state_abbr,under_investigation
0,8001,Adams,2.841043e-01,5.828182,2020-04-03,73,5.471220,0.283264,0.000304,0.317422,0.249106,5.822301,6.0,0.222127,2020-03-28,2020-04-09,66,80,NaN,6.100879,0.078184,0.283264,0.152860,0.086279,CO,1
899,8005,Arapahoe,2.549671e-01,6.464583,2020-04-03,73,6.030084,0.250568,0.000134,0.273260,0.227876,6.433790,6.0,0.178933,2020-03-28,2020-04-09,66,80,NaN,6.688666,0.089308,0.250568,0.125641,0.075161,CO,1
2242,8013,Boulder,1.659277e-01,5.264138,2020-04-03,73,4.820282,0.177496,0.000089,0.195944,0.159048,5.345114,6.0,0.093880,2020-03-28,2020-04-09,66,80,NaN,5.335131,0.067710,0.177496,0.102668,0.050705,CO,1
3721,8019,Clear Creek,-3.140185e-16,1.098612,2020-04-03,73,1.704748,0.057882,0.000157,0.082440,0.033324,1.503786,6.0,0.108041,2020-03-28,2020-04-09,66,80,NaN,1.945910,0.027291,0.057882,0.144074,0.125425,CO,0
5840,8031,Denver,1.864796e-01,6.790154,2020-04-03,73,6.422435,0.193300,0.000338,0.229358,0.157241,6.837894,6.0,0.125971,2020-03-28,2020-04-09,66,80,NaN,6.851185,0.054501,0.193300,0.114224,0.060707,CO,1


In [10]:
decision_space.shape

(23854, 26)

In [11]:
summary_num_investigations = decision_space.groupby("date.y")["under_investigation"].agg(["sum"]).agg(["max","min","mean","median"])
summary_num_investigations

,sum
max,46.000000
min,10.000000
mean,29.995614
median,30.000000


In [12]:
#smallest_decision_space = int(64 - summary_num_investigations.loc["max","sum"])
smallest_decision_space = min(decision_space.groupby("date.y")["under_investigation"].count() - decision_space.groupby("date.y")["under_investigation"].sum())
smallest_decision_space

6

### Are there missing `date.y`

In [13]:
# Continuous dates
v_start = decision_space["days_from_start.y"].min()
v_end = decision_space["days_from_start.y"].max()
continuous_dates = set(range(v_start,v_end+1))
print("Starts on {}, ends on {}, with {} many dates".format(v_start,v_end,len(continuous_dates)))
# Actual dates in decision space
actual_dates = set(decision_space["days_from_start.y"].unique())
print("These are the missing dates: {}".format(continuous_dates - actual_dates))

Starts on 73, ends on 528, with 456 many dates
These are the missing dates: set()


### Generate `EXOG` and `END` (our dataset for VARIMA)

In [14]:
df2 = decision_space.groupby(["date.y"])[["r.grf.y","r.gt.z"]].rank(method="dense",ascending=False)
#df2 = df2.reset_index(drop=True)
df2 = df2.rename(columns={"r.grf.y":"r.grf.y_RANK","r.gt.z":"r.gt.z_RANK"})
VARIMA_decision_space = pd.merge(left=decision_space,right=df2,how="inner",left_index=True,right_index=True)
VARIMA_decision_space.to_csv(os.path.join(OUTPUT_FOLDER,"VARIMA_decision_space.csv"),index=False)

In [15]:
VARIMA_decision_space.head()

,fips,county,r.lm,predicted.lm,date.y,days_from_start.y,log_rolled_cases.y,r.grf,tau.variance,tau.upr,tau.lwr,predicted.grf,date_delta,r.gt.y,date.x,date.z,days_from_start.x,days_from_start.z,log_rolled_cases.x,log_rolled_cases.z,r.gt.z,r.grf.x,r.grf.y,r.grf.z,state_abbr,under_investigation,r.grf.y_RANK,r.gt.z_RANK
0,8001,Adams,2.841043e-01,5.828182,2020-04-03,73,5.471220,0.283264,0.000304,0.317422,0.249106,5.822301,6.0,0.222127,2020-03-28,2020-04-09,66,80,NaN,6.100879,0.078184,0.283264,0.152860,0.086279,CO,1,2.0,4.0
899,8005,Arapahoe,2.549671e-01,6.464583,2020-04-03,73,6.030084,0.250568,0.000134,0.273260,0.227876,6.433790,6.0,0.178933,2020-03-28,2020-04-09,66,80,NaN,6.688666,0.089308,0.250568,0.125641,0.075161,CO,1,7.0,1.0
2242,8013,Boulder,1.659277e-01,5.264138,2020-04-03,73,4.820282,0.177496,0.000089,0.195944,0.159048,5.345114,6.0,0.093880,2020-03-28,2020-04-09,66,80,NaN,5.335131,0.067710,0.177496,0.102668,0.050705,CO,1,13.0,5.0
3721,8019,Clear Creek,-3.140185e-16,1.098612,2020-04-03,73,1.704748,0.057882,0.000157,0.082440,0.033324,1.503786,6.0,0.108041,2020-03-28,2020-04-09,66,80,NaN,1.945910,0.027291,0.057882,0.144074,0.125425,CO,0,3.0,12.0
5840,8031,Denver,1.864796e-01,6.790154,2020-04-03,73,6.422435,0.193300,0.000338,0.229358,0.157241,6.837894,6.0,0.125971,2020-03-28,2020-04-09,66,80,NaN,6.851185,0.054501,0.193300,0.114224,0.060707,CO,1,10.0,7.0


### Begin Statsmodel Rolling Forecast

In [16]:
fips_list = sorted(VARIMA_decision_space["fips"].unique())
dta = VARIMA_decision_space[["date.y","fips","r.grf.y_RANK","r.gt.z_RANK"]]
dta = dta.sort_values(by=["date.y","r.grf.y_RANK"])
#dta = dta[dta["r.grf.y_RANK"] <= smallest_decision_space]

#endog_dta = dta.pivot(index="date.y",columns="fips",values="r.gt.z_RANK")
#endog_dta.index = pd.DatetimeIndex(endog_dta.index).to_period("D")

#exog_dta = dta.pivot(index="date.y",columns="fips",values="r.grf.y_RANK")
#exog_dta.index = pd.DatetimeIndex(exog_dta.index).to_period("D")
list_of_lists = pd.DataFrame()
list_of_lists["date.y"] = sorted(dta["date.y"].unique())
list_of_lists["endog"] = [[np.nan] for d in list_of_lists["date.y"]]
list_of_lists["exog"] = [[np.nan] for d in list_of_lists["date.y"]]
endog = np.zeros((len(list_of_lists.index),int(smallest_decision_space)))
exog = np.zeros((len(list_of_lists.index),int(smallest_decision_space)))

for i,d in enumerate(list_of_lists["date.y"]):
    d_mask = dta["date.y"] == d
    list_of_lists_dmask = list_of_lists["date.y"] == d
    list_of_lists.loc[list_of_lists_dmask,"endog"] = str(dta.loc[d_mask,"r.gt.z_RANK"].tolist())
    list_of_lists.loc[list_of_lists_dmask,"exog"] = str(dta.loc[d_mask,"r.grf.y_RANK"].tolist())
    endog[i,:] = dta.loc[d_mask,"r.gt.z_RANK"].head(n=smallest_decision_space).tolist()
    exog[i,:] = dta.loc[d_mask,"r.grf.y_RANK"].head(n=smallest_decision_space).tolist()

In [17]:
# Value inside each cell is the r.gt.z_RANK
# column states that for that week it was the i-th highest in terms of grf.y_RANK
endog_grf_cols = ["r.grf.y_RANK={}".format(i) for i in range(1,smallest_decision_space+1)]
endog_df = pd.DataFrame(endog, columns=endog_grf_cols)
endog_df["date.y"] = list_of_lists["date.y"] 
endog_df

,r.grf.y_RANK=1,r.grf.y_RANK=2,r.grf.y_RANK=3,r.grf.y_RANK=4,r.grf.y_RANK=5,r.grf.y_RANK=6,date.y
0,2.0,4.0,12.0,11.0,3.0,6.0,2020-04-03
1,4.0,11.0,2.0,3.0,13.0,1.0,2020-04-04
2,5.0,2.0,11.0,4.0,3.0,8.0,2020-04-05
3,1.0,7.0,13.0,5.0,4.0,3.0,2020-04-06
4,1.0,6.0,14.0,5.0,9.0,7.0,2020-04-07
...,...,...,...,...,...,...,...
451,20.0,5.0,21.0,4.0,18.0,28.0,2021-06-28
452,7.0,32.0,3.0,20.0,24.0,9.0,2021-06-29
453,2.0,57.0,9.0,49.0,5.0,44.0,2021-06-30
454,3.0,9.0,6.0,55.0,5.0,44.0,2021-07-01


In [18]:
mod = sm.tsa.VARMAX(endog_df[endog_grf_cols], order=(1,0))
res = mod.fit(maxiter=100, disp=False)
print(res.summary())

                                                                        Statespace Model Results                                                                        
Dep. Variable:     ['r.grf.y_RANK=1', 'r.grf.y_RANK=2', 'r.grf.y_RANK=3', 'r.grf.y_RANK=4', 'r.grf.y_RANK=5', 'r.grf.y_RANK=6']   No. Observations:                  456
Model:                                                                                                                   VAR(1)   Log Likelihood              -10855.603
                                                                                                                    + intercept   AIC                          21837.206
Date:                                                                                                          Tue, 22 Mar 2022   BIC                          22096.923
Time:                                                                                                                  19:38:38   HQIC                     

In [19]:
for endog_grf in endog_grf_cols:
    print(endog_df[endog_df[endog_grf] <7][endog_grf].count()/endog_df.shape[0])

0.5350877192982456
0.4342105263157895
0.4100877192982456
0.3574561403508772
0.2807017543859649
0.26535087719298245


In [20]:
case_study_decision_space_df = pd.read_csv(os.path.join(OUTPUT_FOLDER,"case_study_decision_space.csv"))
case_study_decision_space_df.head()

,fips,county,r.lm,predicted.lm,date.y,days_from_start.y,log_rolled_cases.y,r.grf,tau.variance,tau.upr,tau.lwr,predicted.grf,date_delta,r.gt.y,date.x,date.z,days_from_start.x,days_from_start.z,log_rolled_cases.x,log_rolled_cases.z,r.gt.z,r.grf.x,r.grf.y,r.grf.z,state_abbr,under_investigation,chosen_by_CDPHE,capacity,r.grf.y_RANK,r.gt.z_RANK,chosen_by_TLGRF1,chosen_by_TLGRF2,chosen_by_TLGRF3,chosen_by_TLGRF4,chosen_by_TLGRF5,chosen_by_TLGRF6,chosen_by_TLGRF7,chosen_by_TLGRF8,chosen_by_TLGRF9
0,8014,Broomfield,0.068053,3.420813,2020-04-08,78,3.921973,0.096711,0.000453,0.138432,0.054989,3.621414,11.0,0.141668,2020-04-02,2020-04-14,71,85,2.944439,4.421848,0.076995,0.096711,0.048230,0.037621,CO,0,0,1,3.0,1.0,0,0,1,0,0,0,0,0,0
1,8019,Clear Creek,0.207639,2.839770,2020-04-08,78,1.658228,0.174502,0.000139,0.197619,0.151384,2.607806,11.0,0.004151,2020-04-02,2020-04-14,71,85,1.386294,2.079442,0.045858,0.174502,0.015569,-0.017276,CO,0,0,1,9.0,3.0,0,0,0,0,0,0,0,0,1
2,8037,Eagle,0.129325,6.337903,2020-04-08,78,5.805135,0.123289,0.000049,0.136969,0.109609,6.295652,11.0,0.036743,2020-04-02,2020-04-14,71,85,5.432630,5.936876,0.017369,0.123289,0.032999,0.018065,CO,0,0,1,7.0,7.0,0,0,0,0,0,0,1,0,0
3,8039,Elbert,0.125163,2.323061,2020-04-08,78,2.224624,0.140186,0.000281,0.173047,0.107324,2.428220,11.0,0.119824,2020-04-02,2020-04-14,71,85,1.446919,2.583998,0.057889,0.140186,0.040725,-0.013338,CO,0,0,1,5.0,2.0,0,0,0,0,1,0,0,0,0
4,8041,El Paso,0.131502,6.495518,2020-04-08,78,6.101439,0.136595,0.000024,0.146182,0.127009,6.531168,11.0,0.068981,2020-04-02,2020-04-14,71,85,5.575002,6.359141,0.035760,0.136595,0.055108,0.022869,CO,0,0,1,2.0,4.0,0,1,0,0,0,0,0,0,0


In [21]:
CDPHE_ranks = case_study_decision_space_df[case_study_decision_space_df["chosen_by_CDPHE"] == 1][["date.y","r.gt.z_RANK"]]
CDPHE_ranks = pd.DataFrame(CDPHE_ranks)
CDPHE_ranks

,date.y,r.gt.z_RANK
7,2020-04-08,9.0
13,2020-04-20,6.0
61,2020-04-27,5.0
64,2020-05-05,1.0
90,2020-05-07,12.0
...,...,...
2088,2021-06-06,8.0
2110,2021-06-15,4.0
2130,2021-06-21,22.0
2164,2021-06-23,16.0


In [22]:
best_k = 1
print("Counting TP as the chosen county being in the top {}".format(best_k))
CDPHE_TP = CDPHE_ranks[CDPHE_ranks["r.gt.z_RANK"] == best_k]["r.gt.z_RANK"].count()
CDPHE_P = CDPHE_ranks.shape[0]
CDPHE_PPV = CDPHE_TP/CDPHE_P
print("CDPHE TP={}, P={}, PPV={}".format(CDPHE_TP,CDPHE_P,CDPHE_PPV))
for n,TLGRF in enumerate(["chosen_by_TLGRF{}".format(i) for i in range(1,10)]):
    TLGRF_rank = case_study_decision_space_df[case_study_decision_space_df[TLGRF] == 1][["date.y","r.gt.z_RANK"]]
    TLGRF_TP = TLGRF_rank[TLGRF_rank["r.gt.z_RANK"] == best_k]["r.gt.z_RANK"].count()
    TLGRF_P = TLGRF_rank.shape[0]
    TLGRF_PPV = TLGRF_TP/TLGRF_P
    print("TLGRF{} TP={}, P={}, PPV={}".format(n+1,TLGRF_TP,TLGRF_P,TLGRF_PPV))

Counting TP as the chosen county being in the top 1
CDPHE TP=13, P=117, PPV=0.1111111111111111
TLGRF1 TP=8, P=117, PPV=0.06837606837606838
TLGRF2 TP=10, P=117, PPV=0.08547008547008547
TLGRF3 TP=10, P=117, PPV=0.08547008547008547
TLGRF4 TP=7, P=117, PPV=0.05982905982905983
TLGRF5 TP=5, P=117, PPV=0.042735042735042736
TLGRF6 TP=7, P=117, PPV=0.05982905982905983
TLGRF7 TP=6, P=117, PPV=0.05128205128205128
TLGRF8 TP=4, P=117, PPV=0.03418803418803419
TLGRF9 TP=4, P=117, PPV=0.03418803418803419
